# Step 4c: Recheck Corrected & Custom-Generated R1 Rows

**Purpose:** One-time standalone recheck for the 43 R1 rows that went through a
one-way correction in `log-gen-verifier.ipynb` Steps 4a/4b.

| Source | Rows | Risk |
|---|---|---|
| `Correction_Mode == 'Targeted'` | ~41 | Fixing one criterion may silently break another |
| `Correction_Mode == 'Full_Custom'` | ~2 | Custom generation was never independently scored |

**Inputs:**
- `Datasets/verified/verified_r1_seeds_combined.csv` — 225 verified R1 dialogues (corrected dialogue source)
- `Datasets/verified/verified_r1_criteria_scores.csv` — per-criterion scores with `Correction_Mode` column
- `Datasets/seeds/generated_r1_seeds_gemini3.csv` — seed CSV (authoritative game context: player_roles, public_history)

**Outputs (updated in-place):**
- `Datasets/verified/verified_r1_seeds_combined.csv` — patched `discussion_log`, `matrix_tactic_scale`, `LLM_used` for changed rows
- `Datasets/verified/verified_r1_criteria_scores.csv` — new columns: `Recheck_4c`, `Recheck_4c_Score`, `Recheck_4c_Reasoning`, `Recheck_4c_Explanations`

**Recheck decision logic:**

| Score | Action | Tag |
|---|---|---|
| 5/5 | Accept as-is | `ACCEPTED` |
| 4/5 | Targeted correction of failing criterion | `RECORRECTED` |
| ≤3/5 | Full regeneration | `REGENERATED` |

## Imports & File Paths

In [13]:
import pandas as pd
import json
import os
import re
from anthropic import Anthropic
from dotenv import load_dotenv
import time

GEMINI3_FILE         = "Datasets/seeds/generated_r1_seeds_gemini3.csv"
OUTPUT_FILE_COMBINED = "Datasets/verified/verified_r1_seeds_combined.csv"
OUTPUT_FILE_CRITERIA = "Datasets/verified/verified_r1_criteria_scores.csv"
TACTICS_KB_FILE      = "tactics_knowledge_base.json"

print("✓ Imports done")
for f in [GEMINI3_FILE, OUTPUT_FILE_COMBINED, OUTPUT_FILE_CRITERIA, TACTICS_KB_FILE]:
    status = "✓" if os.path.exists(f) else "✗ MISSING"
    print(f"  {status}  {f}")

✓ Imports done
  ✓  generated_r1_seeds_gemini3.csv
  ✓  verified_r1_seeds_combined.csv
  ✓  verified_r1_criteria_scores.csv
  ✓  tactics_knowledge_base.json


## Load Data & Initialize Claude Client

In [14]:

# Load CSVs
gemini3_df  = pd.read_csv(GEMINI3_FILE)
combined_df = pd.read_csv(OUTPUT_FILE_COMBINED)
criteria_df = pd.read_csv(OUTPUT_FILE_CRITERIA, dtype=str).fillna('')

print(f"✓ gemini3_df:   {gemini3_df.shape[0]} rows")
print(f"✓ combined_df:  {combined_df.shape[0]} rows")
print(f"✓ criteria_df:  {criteria_df.shape[0]} rows")

# Build tactic taxonomy reference
with open(TACTICS_KB_FILE, 'r') as _f:
    _kb = json.load(_f)

_COLOR_LABEL = {'green': 'Good-only', 'red': 'Evil-only', 'blue': 'Both alignments'}
_COLOR_ORDER = ['green', 'red', 'blue']
_grouped = {c: [] for c in _COLOR_ORDER}
for cell_key, cell in _kb['behavior_matrix'].items():
    row, col = cell_key.split('|')
    for tactic in cell['tactics']:
        _grouped[cell['color']].append((row, col, tactic))

_scale_lines = []
for scale_key, sd in _kb['scale_definitions'].items():
    level_strs = '; '.join(f'{lvl}: {desc}' for lvl, desc in sd['levels'].items())
    _scale_lines.append(f'  {scale_key} ({sd["name"]}) — for {sd["for"]}: {level_strs}')

_tax_lines = ['TACTIC ANNOTATION TAXONOMY']
_tax_lines.append('')
_tax_lines.append('The 4×4 Behavioral Matrix has two dimensions:')
_tax_lines.append('  Row (Information Strategy): Transparent | Selective/Framing | Careless-to-truth | Counterfactual')
_tax_lines.append('    - Transparent: speaker openly shares clear evidence-based information')
_tax_lines.append('    - Selective/Framing: speaker shapes communication through selective disclosure, emphasis, or framing')
_tax_lines.append('    - Careless-to-truth: speaker is indifferent to factual accuracy, i.e., bullshitting, vague speculation, or noise')
_tax_lines.append('    - Counterfactual: speaker asserts or believes information contrary to reality, i.e., honest mistake, self-deception, or hard lying')
_tax_lines.append('  Column (Goal Orientation): Cooperative | Defensive | Opportunistic | Adversarial')
_tax_lines.append('    - Cooperative: helps the group reach accurate conclusions')
_tax_lines.append('    - Defensive: protects self from suspicion or blame')
_tax_lines.append('    - Opportunistic: exploits the social moment for personal or team advantage')
_tax_lines.append('    - Adversarial: actively undermines specific players or misdirects the group')
_tax_lines.append('')
_tax_lines.append('Valid tactic names by alignment (use ONLY these exact names):')
for color in _COLOR_ORDER:
    _tax_lines.append(f'  {_COLOR_LABEL[color]}:')
    for row, col, tactic in _grouped[color]:
        _tax_lines.append(f'    [{row} | {col}] {tactic}')
_tax_lines.append('')
_tax_lines.append('Skill scales ("scale" field in matrix_tactic_scale):')
_tax_lines.extend(_scale_lines)
_tax_lines.append('  Note: Good players at Moderate/Low skill may legitimately use Blue (Both-alignment) tactics.')
_tax_lines.append('  Do NOT penalize a Good player for using Deflection, Self-deception, Face-saving blather,')
_tax_lines.append('  Vague filler, Noisy sincerity, Self-justification, Pragmatic silence, Omission to save face,')
_tax_lines.append('  Half-truths, or Rationalized error — these are valid defensive behaviors for Good players.')

TACTIC_TAXONOMY_REF = '\n'.join(_tax_lines)
print(f"\n✓ TACTIC_TAXONOMY_REF built ({len(_grouped['green']) + len(_grouped['red']) + len(_grouped['blue'])} tactics)")
# print("\n--- TACTIC_TAXONOMY_REF ---")
# print(TACTIC_TAXONOMY_REF)
# print("--- END TACTIC_TAXONOMY_REF ---")

# Build the static evaluation prompt — evaluated once, cached across all 43 API calls.
# Must be defined here (after TACTIC_TAXONOMY_REF) and placed FIRST in the messages
# content list so the cache prefix is stable. The dynamic per-row context follows it.
STATIC_EVAL_PROMPT = f"""**TACTIC ANNOTATION FRAMEWORK:**
{TACTIC_TAXONOMY_REF}

Each speaker's annotation in matrix_tactic_scale must use this structure:
  "PX": {{"row": "<Information Strategy>", "col": "<Goal Orientation>", "tactic": "<exact name from taxonomy>", "scale": "GRS|Mach-IV", "level": "High|Moderate|Low"}}

---

**EVALUATION CRITERIA (1 = PASS, 0 = FAIL):**

1. **Discussion Coherence & Tactic Situational Fit**:
   1 = Each player's tactic makes situational sense given the social dynamics of the discussion — their conversational position, what other players are saying, and how tensions develop across the exchange
   0 = A player's tactic is socially incongruous in a severe and objectively clear way (e.g., "Pragmatic silence" for a player being directly accused by multiple others; "Vague filler" for a player everyone is turning to for information). The utterance may match the label literally, but the tactic choice is objectively implausible for that social moment.
   **Benefit of the doubt**: Borderline cases, edge-case tactic choices, and ambiguous social contexts must score 1. Only fail for clear, severe impossibilities where no other interpretation is possible.

2. **Public History Alignment**:
   1 = Dialogue contextually fits quest outcomes, team compositions, and voting patterns
   0 = Ignores prior events or directly contradicts an established fact (e.g., names a player who wasn't on the team, states a quest passed when it failed)
   **Benefit of the doubt**: Minor omissions or vague references to prior game state are acceptable. Only fail when the dialogue contains a specific, verifiable factual contradiction with the public history.

3. **Tactic-Dialogue Alignment**:
   1 = Each player's annotation accurately describes what they actually said: the tactic name is a valid entry from the taxonomy, the row/col match the tactic's position in the 4×4 matrix, scale is GRS for Good players and Mach-IV for Evil players, and the level (High/Moderate/Low) is plausible given the sophistication of the utterance
   0 = The annotation has a clear, objective error: a tactic name NOT in the valid taxonomy, wrong scale for the player's role (Evil with GRS or Good with Mach-IV), wrong row/col for the named tactic, OR the tactic bears no plausible relationship whatsoever to what the player said.
   Framing: the dialogue is the ground truth — the annotation must describe what was written.
   **Benefit of the doubt**: When multiple tactic interpretations are plausible for an utterance and the assigned tactic is one valid reading of it, score 1. Do NOT fail for alternative-but-valid interpretations, minor label imprecisions, or cases where you would personally choose a different (but also valid) tactic. Only fail for objectively incorrect annotations.

4. **Authenticity & Skill Consistency**:
   1 = The dialogue reads as a believable Avalon exchange with natural accusation/defense patterns, and the linguistic sophistication of each utterance is consistent with the speaker's assigned skill level.
   0 = The dialogue feels stilted or artificial, or the sophistication of an utterance clearly mismatches the assigned level (e.g., elaborate strategic reasoning from a Low-skill player, or uncertain hedging from a High-skill player).
   **Benefit of the doubt**: Minor awkwardness, slightly formal phrasing, or marginal skill-level imprecision must score 1. Only fail when inauthenticity is pervasive throughout the dialogue, or a skill-level mismatch is severe and unambiguous.

5. **Dialogue Format Compliance**:
   1 = Starts with "Discussion after Quest X:", exactly 4 speakers (all non-protagonists), each speaks exactly once, in this format:
      Discussion after Quest X:
      PlayerA: [dialogue]
      PlayerB: [dialogue]
      PlayerC: [dialogue]
      PlayerD: [dialogue]
   0 = Wrong or missing header, incorrect speaker count, protagonist appears as speaker, or any format violations
   **Benefit of the doubt**: Minor formatting differences (e.g., extra blank lines between speakers, slight whitespace variation) must score 1. Only fail for clear structural violations: missing/wrong header, wrong speaker count, or protagonist speaking.

---

**DECISION LOGIC:**

**Your default is ACCEPT.** This dialogue has already been corrected once. Only flag a criterion as failing when the flaw is clear, objective, and non-debatable. Subjective interpretation differences, minor annotation imprecisions, and edge-case tactic choices are NOT failures. When uncertain whether something is a flaw, score it as passing (1).

1. **5/5** → ACCEPT the dialogue as-is (the expected outcome for most dialogues in this second-pass evaluation)
2. **4/5** → TARGETED_CORRECTION: fix the single failing criterion — only when the failure is objectively clear and unambiguous
3. **≤3/5** → FULL_REGENERATION: generate a completely new dialogue that passes all 5 criteria

---

**REQUIRED OUTPUT FORMAT:**

Return a valid JSON object. For most dialogues in this second-pass evaluation, the expected output is a full 5/5 ACCEPT:

```json
{{
  "criteria": {{
    "coherence": {{"score": 1, "explanation": "Brief explanation why it passed"}},
    "history": {{"score": 1, "explanation": "Brief explanation why it passed"}},
    "matrix": {{"score": 1, "explanation": "Brief explanation why it passed"}},
    "authenticity": {{"score": 1, "explanation": "Brief explanation why it passed"}},
    "format": {{"score": 1, "explanation": "Brief explanation why it passed"}}
  }},
  "total": 5,
  "recommendation": "ACCEPT",
  "reasoning": "2-3 sentences on overall quality",
  "failed_criterion": null,
  "targeted_correction": null,
  "regenerated_dialogue": null
}}
```

If exactly one criterion has a clear, objective failure (4/5), use TARGETED_CORRECTION:

```json
{{
  "criteria": {{
    "coherence": {{"score": 1, "explanation": "Brief explanation why it passed or failed"}},
    "history": {{"score": 1, "explanation": "Brief explanation why it passed or failed"}},
    "matrix": {{"score": 0, "explanation": "Brief explanation of the objective failure"}},
    "authenticity": {{"score": 1, "explanation": "Brief explanation why it passed or failed"}},
    "format": {{"score": 1, "explanation": "Brief explanation why it passed or failed"}}
  }},
  "total": 4,
  "recommendation": "TARGETED_CORRECTION",
  "reasoning": "2-3 sentences on overall quality and decision",
  "failed_criterion": "Matrix",
  "targeted_correction": {{
    "dialogue": "Full corrected dialogue text fixing only the failed criterion",
    "matrix_tactic_scale": {{"PX": {{"row": "...", "col": "...", "tactic": "...", "scale": "GRS|Mach-IV", "level": "High|Moderate|Low"}}, "...": {{...}}}}
  }},
  "regenerated_dialogue": null
}}
```

When recommendation is FULL_REGENERATION, regenerated_dialogue takes this structure (targeted_correction will be null):

```json
{{
  ...,
  "recommendation": "FULL_REGENERATION",
  "targeted_correction": null,
  "regenerated_dialogue": {{
    "dialogue": "Discussion after Quest X:\\nPlayerA: [utterance]\\nPlayerB: [utterance]\\nPlayerC: [utterance]\\nPlayerD: [utterance]",
    "matrix_tactic_scale": {{"PX": {{"row": "...", "col": "...", "tactic": "...", "scale": "GRS|Mach-IV", "level": "High|Moderate|Low"}}, "...": {{...}}}}
  }}
}}
```

**Field rules:**
- **targeted_correction**: Populated only when recommendation is TARGETED_CORRECTION; an object with:
  - "dialogue": full corrected dialogue text fixing only the failed criterion
  - "matrix_tactic_scale": JSON mapping each speaker to their tactic annotation; keep all speakers' entries from the input unless the correction strictly requires a change
  Otherwise null.
- **regenerated_dialogue**: Populated only when recommendation is FULL_REGENERATION; an object with:
  - "dialogue": complete new dialogue (exactly 4 speakers, protagonist excluded, "Discussion after Quest X:" format)
  - "matrix_tactic_scale": JSON mapping each speaker to their tactic annotation; use the input matrix_tactic_scale as baseline — keep each speaker's row, col, tactic, scale, and level unless the new dialogue strictly requires a change
  Otherwise null.
- **failed_criterion**: The single failing criterion name (Coherence/History/Matrix/Authenticity/Format) when recommendation is TARGETED_CORRECTION; null otherwise
- Return ONLY the JSON object, no markdown code blocks, no extra text."""
print(f"✓ STATIC_EVAL_PROMPT built ({len(STATIC_EVAL_PROMPT)} chars)")

# Initialize Claude client
load_dotenv(override=True)
client = Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'), max_retries=5)

SYSTEM_PROMPT = """You are an expert quality verifier for the Avalon social deception game. Your task is to evaluate a single AI-generated dialogue log that has already undergone one round of correction. Score it against 5 binary criteria, and apply the minimum necessary fix only when a criterion has a clear, objective, non-debatable violation. When in doubt, accept the dialogue as-is."""

print("✓ Claude client initialized")


✓ gemini3_df:   225 rows
✓ combined_df:  225 rows
✓ criteria_df:  225 rows

✓ TACTIC_TAXONOMY_REF built (37 tactics)
✓ STATIC_EVAL_PROMPT built (12303 chars)
✓ Claude client initialized


## Helper Functions

In [16]:
_CRITERIA_ORDER = ['coherence', 'history', 'matrix', 'authenticity', 'format']


def reorder_matrix_by_speaking_order(dialogue: str, matrix_input) -> str:
    """Reorder matrix_tactic_scale keys to match speaking order in the dialogue."""
    try:
        speakers = re.findall(r'(?m)^(P\d+):', dialogue or '')
        seen, ordered_speakers = set(), []
        for s in speakers:
            if s not in seen:
                seen.add(s)
                ordered_speakers.append(s)
        matrix = json.loads(matrix_input) if isinstance(matrix_input, str) else (matrix_input or {})
        reordered = {s: matrix[s] for s in ordered_speakers if s in matrix}
        for k, v in matrix.items():
            if k not in reordered:
                reordered[k] = v
        lines = [f'  "{k}": ' + json.dumps(v, separators=(',', ':')) for k, v in reordered.items()]
        return '{\n' + ',\n'.join(lines) + '\n}'
    except Exception:
        if isinstance(matrix_input, str):
            return matrix_input
        m = matrix_input or {}
        lines = [f'  "{k}": ' + json.dumps(v, separators=(',', ':')) for k, v in m.items()]
        return '{\n' + ',\n'.join(lines) + '\n}'


print("✓ Helper functions defined")

✓ Helper functions defined


## `verify_single_dialogue()` Function

In [17]:

def verify_single_dialogue(game_id, round_id, player_roles, protagonist,
                            public_history, discussion_log, matrix_tactic_scale):
    # Dynamic part: game context + dialogue to evaluate (unique per row, never cached)
    # STATIC_EVAL_PROMPT (taxonomy + criteria + output format) is sent as a separate
    # content block with cache_control, so it's only processed once across all 43 calls.
    dynamic_part = f"""You are evaluating a single AI-generated Avalon discussion log for quality. The dialogue has already undergone one round of correction. Your default is to ACCEPT it — only flag a criterion as failing when there is a clear, objective, non-debatable violation.

**CONTEXT:**
Game: {game_id}, Round {round_id}
Player Roles: {player_roles}
Protagonist (does NOT speak in dialogue): {protagonist}

Public History (up to current round):
{public_history}

---

**DIALOGUE TO EVALUATE:**

Discussion Log:
{discussion_log}

Matrix Tactic Scale:
{matrix_tactic_scale}

---
"""

    try:
        response = client.messages.create(
            model="claude-sonnet-4-5-20250929",
            max_tokens=4096,
            system=SYSTEM_PROMPT,
            messages=[{
                "role": "user",
                "content": [
                    # Static block first: taxonomy + criteria + output format.
                    # cache_control marks where the cache prefix ends — written once,
                    # then read at 10% input price on all subsequent calls.
                    {"type": "text", "text": STATIC_EVAL_PROMPT, "cache_control": {"type": "ephemeral"}},
                    # Dynamic block second: game context + dialogue (unique, never cached)
                    {"type": "text", "text": dynamic_part},
                ]
            }]
        )
        response_text = response.content[0].text

        try:
            clean_text = response_text.strip()
            if clean_text.startswith('```'):
                lines = clean_text.split('\n')
                lines = lines[1:]
                for i, line in enumerate(lines):
                    if line.strip() == '```':
                        lines = lines[:i]
                        break
                clean_text = '\n'.join(lines)

            data     = json.loads(clean_text)
            criteria = data['criteria']

            result = {
                'criteria': {k: criteria[k]['score'] == 1 for k in _CRITERIA_ORDER},
                'criteria_explanations': {k: criteria[k].get('explanation', '') for k in _CRITERIA_ORDER},
                'total': int(data['total']) if isinstance(data['total'], int) else int(str(data['total']).split('/')[0]),
                'recommendation':    data.get('recommendation', 'ACCEPT'),
                'reasoning':         data.get('reasoning', ''),
                'failed_criterion':  data.get('failed_criterion'),
                'targeted_correction': data.get('targeted_correction'),
                'regenerated_dialogue': data.get('regenerated_dialogue'),
                'selected_dialogue':     None,
                'selected_tactic_scale': None,
                'correction_mode':       None,
                'needs_retry':           False,
            }

            rec = result['recommendation']

            if result['total'] == 5 or rec == 'ACCEPT':
                result['correction_mode']       = 'Accepted'
                result['selected_dialogue']     = discussion_log
                result['selected_tactic_scale'] = reorder_matrix_by_speaking_order(discussion_log, matrix_tactic_scale)

            elif rec == 'TARGETED_CORRECTION':
                tc          = result.get('targeted_correction') or {}
                tc_dialogue = tc.get('dialogue') if isinstance(tc, dict) else None
                tc_matrix   = tc.get('matrix_tactic_scale') if isinstance(tc, dict) else None
                if tc_dialogue is None:
                    print(f"  ⚠️  WARNING: targeted_correction.dialogue is None for {game_id} — flagging for retry")
                    result['needs_retry'] = True
                result['correction_mode']       = 'Recorrected'
                result['selected_dialogue']     = tc_dialogue or discussion_log
                result['selected_tactic_scale'] = reorder_matrix_by_speaking_order(
                    tc_dialogue or discussion_log,
                    json.dumps(tc_matrix, separators=(',', ':')) if isinstance(tc_matrix, dict) else (tc_matrix or matrix_tactic_scale)
                )

            else:  # FULL_REGENERATION
                rd          = result.get('regenerated_dialogue') or {}
                rd_dialogue = rd.get('dialogue') if isinstance(rd, dict) else None
                rd_matrix   = rd.get('matrix_tactic_scale') if isinstance(rd, dict) else None
                if not rd_dialogue:
                    print(f"  ⚠️  WARNING: regenerated_dialogue is empty for {game_id}!")
                    result['needs_retry'] = True
                result['correction_mode']       = 'Regenerated'
                result['selected_dialogue']     = rd_dialogue or discussion_log
                result['selected_tactic_scale'] = reorder_matrix_by_speaking_order(
                    rd_dialogue or discussion_log,
                    json.dumps(rd_matrix, separators=(',', ':')) if isinstance(rd_matrix, dict) else (rd_matrix or matrix_tactic_scale)
                )

            return result, response_text

        except json.JSONDecodeError as e:
            print(f"❌ JSON parsing failed for {game_id}: {e}")
            return None, f"JSON parsing error: {e}"
        except KeyError as e:
            print(f"❌ Missing JSON field for {game_id}: {e}")
            return None, f"Missing field: {e}"
        except Exception as e:
            print(f"❌ Error processing recheck JSON for {game_id}: {e}")
            return None, str(e)

    except Exception as e:
        print(f"  Error rechecking game {game_id}: {str(e)}")
        return None, str(e)


print("✓ verify_single_dialogue() defined")


✓ verify_single_dialogue() defined


## Filter Recheck Rows

In [18]:
# Build list of criteria dicts (working copy — we'll patch these then write back)
criteria_records = criteria_df.to_dict('records')

recheck_rows = [
    cr for cr in criteria_records
    if cr.get('Correction_Mode') in ('Targeted', 'Full_Custom')
]

n_targeted = sum(1 for cr in recheck_rows if cr['Correction_Mode'] == 'Targeted')
n_custom   = sum(1 for cr in recheck_rows if cr['Correction_Mode'] == 'Full_Custom')

print(f"Rows to recheck: {len(recheck_rows)} total")
print(f"  Targeted:    {n_targeted}")
print(f"  Full_Custom: {n_custom}")
print()

if not recheck_rows:
    print("✅ No corrected rows found — nothing to recheck.")
else:
    print("Game IDs scheduled for recheck:")
    for cr in recheck_rows:
        print(f"  {cr['ID']}  ({cr['Correction_Mode']})  LLM: {cr['Selected_LLM']}")

Rows to recheck: 43 total
  Targeted:    41
  Full_Custom: 2

Game IDs scheduled for recheck:
  G028/1  (Targeted)  LLM: Gemini-3.1-Claude-4.5
  G035/1  (Targeted)  LLM: GPT-5.2-Claude-4.5
  G043/1  (Targeted)  LLM: GPT-5.2-Claude-4.5
  G048/1  (Full_Custom)  LLM: Claude-4.5
  G055/1  (Targeted)  LLM: GPT-5.2-Claude-4.5
  G058/1  (Targeted)  LLM: Gemini-3.1-Claude-4.5
  G063/1  (Targeted)  LLM: GPT-5.2-Claude-4.5
  G064/1  (Targeted)  LLM: GPT-5.2-Claude-4.5
  G065/1  (Targeted)  LLM: GPT-5.2-Claude-4.5
  G085/1  (Targeted)  LLM: GPT-5.2-Claude-4.5
  G091/1  (Full_Custom)  LLM: Claude-4.5
  G097/1  (Targeted)  LLM: GPT-5.2-Claude-4.5
  G099/1  (Targeted)  LLM: GPT-5.2-Claude-4.5
  G102/1  (Targeted)  LLM: Gemini-3.1-Claude-4.5
  G113/1  (Targeted)  LLM: GPT-5.2-Claude-4.5
  G123/1  (Targeted)  LLM: GPT-5.2-Claude-4.5
  G126/1  (Targeted)  LLM: GPT-5.2-Claude-4.5
  G129/1  (Targeted)  LLM: GPT-5.2-Claude-4.5
  G130/1  (Targeted)  LLM: GPT-5.2-Claude-4.5
  G133/1  (Targeted)  LLM: GPT-5.

## Main Recheck Loop

In [ ]:
# Working copy of combined_df rows as list of dicts (patched in-place)
verified_records = combined_df.to_dict('records')

accepted    = 0
recorrected = 0
regenerated = 0
still_failed = 0

for cr in recheck_rows:
    score_id              = cr['ID']
    game_id, round_id_str = score_id.split('/')
    round_id              = int(round_id_str)

    # Game context from seed CSV (authoritative source)
    row_mask = gemini3_df['game_id'] == game_id
    if not row_mask.any():
        print(f"  ⚠️  {score_id}: game not found in gemini3_df — skipping")
        continue
    idx            = gemini3_df.index[row_mask][0]
    player_roles   = gemini3_df.loc[idx, 'player_roles']
    protagonist    = gemini3_df.loc[idx, 'role_id']
    public_history = gemini3_df.loc[idx, 'public_history']

    # Current (already-corrected) dialogue from verified_records
    vr_pos = next(
        (i for i, r in enumerate(verified_records) if r.get('game_id') == game_id),
        None
    )
    if vr_pos is None:
        print(f"  ⚠️  {score_id}: not found in verified_records — skipping")
        continue
    current_dialogue = verified_records[vr_pos]['discussion_log']
    current_matrix   = verified_records[vr_pos]['matrix_tactic_scale']

    print(f"  Rechecking {score_id}  (was: {cr['Correction_Mode']}, LLM: {cr['Selected_LLM']})...")

    result, raw_response = verify_single_dialogue(
        game_id, round_id, player_roles, protagonist, public_history,
        current_dialogue, current_matrix
    )

    # ── API error ────────────────────────────────────────────────────────────
    if result is None:
        print(f"  ✗ Recheck API error for {score_id}: {raw_response}")
        cr['Recheck_4c']              = 'API_ERROR'
        cr['Recheck_4c_Score']        = '0 of 5'
        cr['Recheck_4c_Reasoning']    = f'(api_error: {raw_response})'
        cr['Recheck_4c_Explanations'] = '{}'
        still_failed += 1
        print()
        time.sleep(1)
        continue

    recheck_score = result['total']
    mode          = result['correction_mode']   # 'Accepted' | 'Recorrected' | 'Regenerated'

    # Write recheck metadata into criteria record (in-place)
    cr['Recheck_4c']              = mode.upper()
    cr['Recheck_4c_Score']        = f"{recheck_score} of 5"
    cr['Recheck_4c_Reasoning']    = result['reasoning']
    cr['Recheck_4c_Explanations'] = json.dumps(result['criteria_explanations'])

    pass_str = ', '.join(k for k in _CRITERIA_ORDER if result['criteria'].get(k))
    print(f"  ✓ Recheck score: {recheck_score}/5 ({pass_str}) → {mode}")

    if mode == 'Accepted':
        accepted += 1
        # No change needed; dialogue already correct

    else:
        old_llm    = verified_records[vr_pos].get('LLM_used', '?')
        llm_suffix = '-Recorrected' if mode == 'Recorrected' else '-Regenerated'
        new_llm    = old_llm + llm_suffix

        verified_records[vr_pos]['discussion_log']      = result['selected_dialogue']
        verified_records[vr_pos]['matrix_tactic_scale'] = result['selected_tactic_scale']
        verified_records[vr_pos]['LLM_used']            = new_llm
        verified_records[vr_pos]['Claude_Reasoning']    = (
            str(verified_records[vr_pos].get('Claude_Reasoning', ''))
            + f" | 4c: {result['reasoning']}"
        )
        cr['Selected_LLM'] = new_llm

        print(f"  ✓ Patched verified_records [pos {vr_pos}]: {old_llm} → {new_llm}")
        if mode == 'Recorrected':
            recorrected += 1
        else:
            regenerated += 1

    print()
    time.sleep(1)

print("─" * 60)
print(f"✅ Recheck loop complete:")
print(f"   Accepted  (5/5, no change):   {accepted}")
print(f"   Recorrected (4/5 targeted):   {recorrected}")
print(f"   Regenerated (≤3/5 regen):     {regenerated}")
print(f"   Still failed (API error):     {still_failed}")
print(f"   Total processed: {accepted + recorrected + regenerated + still_failed} / {len(recheck_rows)}")

  Rechecking G028/1  (was: Targeted, LLM: Gemini-3.1-Claude-4.5)...
  ✓ Recheck score: 5/5 (coherence, history, matrix, authenticity, format) → Accepted

  Rechecking G035/1  (was: Targeted, LLM: GPT-5.2-Claude-4.5)...
  ✓ Recheck score: 4/5 (coherence, history, authenticity, format) → Recorrected
  ✓ Patched verified_records [pos 9]: GPT-5.2-Claude-4.5 → GPT-5.2-Claude-4.5-4cFixed

  Rechecking G043/1  (was: Targeted, LLM: GPT-5.2-Claude-4.5)...
  ✓ Recheck score: 5/5 (coherence, history, matrix, authenticity, format) → Accepted

  Rechecking G048/1  (was: Full_Custom, LLM: Claude-4.5)...
  ✓ Recheck score: 3/5 (history, authenticity, format) → Regenerated
  ✓ Patched verified_records [pos 22]: Claude-4.5 → Claude-4.5-4cRegen

  Rechecking G055/1  (was: Targeted, LLM: GPT-5.2-Claude-4.5)...
  ✓ Recheck score: 4/5 (coherence, history, authenticity, format) → Recorrected
  ✓ Patched verified_records [pos 29]: GPT-5.2-Claude-4.5 → GPT-5.2-Claude-4.5-4cFixed

  Rechecking G058/1  (was: Ta

## Review: Old vs New

Run this cell after the loop completes. It loads the **original on-disk CSV** and compares it against `verified_records` (in memory) for every row that actually changed. Review before running the Save cell.

In [24]:
import textwrap

# Load original CSV from disk to compare against in-memory verified_records
_orig_df     = pd.read_csv(OUTPUT_FILE_COMBINED)
_orig_by_gid = {r['game_id']: r for r in _orig_df.to_dict('records')}
_new_by_gid  = {r['game_id']: r for r in verified_records}

# Only examine rows that changed (mode != Accepted)
_changed_ids = [
    cr['ID'].split('/')[0]
    for cr in criteria_records
    if cr.get('Recheck_4c') in ('RECORRECTED', 'REGENERATED')
]

_still_api_err = [
    cr['ID'].split('/')[0]
    for cr in criteria_records
    if cr.get('Recheck_4c') == 'API_ERROR'
]

print(f"Changed rows: {len(_changed_ids)}  |  API errors: {len(_still_api_err)}")
print(f"Accepted (no change): {sum(1 for cr in criteria_records if cr.get('Recheck_4c') == 'ACCEPTED')}")
print("=" * 80)

for game_id in _changed_ids:
    cr = next((c for c in criteria_records if c['ID'].startswith(game_id + '/')), {})
    old_r = _orig_by_gid.get(game_id, {})
    new_r = _new_by_gid.get(game_id, {})

    print(f"\n{'━'*80}")
    print(f"  Game: {game_id}   Mode: {cr.get('Recheck_4c')}   Score: {cr.get('Recheck_4c_Score')}")
    print(f"  Old LLM: {old_r.get('LLM_used','?')}  →  New LLM: {new_r.get('LLM_used','?')}")
    print(f"  Reasoning: {cr.get('Recheck_4c_Reasoning','')}")

    old_diag = str(old_r.get('discussion_log', ''))
    new_diag = str(new_r.get('discussion_log', ''))
    old_mat  = str(old_r.get('matrix_tactic_scale', ''))
    new_mat  = str(new_r.get('matrix_tactic_scale', ''))

    dialogue_changed = old_diag.strip() != new_diag.strip()
    matrix_changed   = old_mat.strip() != new_mat.strip()

    print(f"\n  [DIALOGUE {'CHANGED' if dialogue_changed else 'UNCHANGED'}]")
    if dialogue_changed:
        print("\n  ── OLD ──")
        for line in old_diag.strip().splitlines():
            print(f"    {line}")
        print("\n  ── NEW ──")
        for line in new_diag.strip().splitlines():
            print(f"    {line}")
    else:
        print("  (dialogue identical — only matrix annotation changed)")

    print(f"\n  [MATRIX {'CHANGED' if matrix_changed else 'UNCHANGED'}]")
    if matrix_changed:
        print("\n  ── OLD MATRIX ──")
        for line in old_mat.strip().splitlines():
            print(f"    {line}")
        print("\n  ── NEW MATRIX ──")
        for line in new_mat.strip().splitlines():
            print(f"    {line}")

print(f"\n{'='*80}")
print("Review complete. Run the Save cell when satisfied.")

Changed rows: 22  |  API errors: 0
Accepted (no change): 21

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Game: G035   Mode: RECORRECTED   Score: 4 of 5
  Old LLM: GPT-5.2-Claude-4.5  →  New LLM: GPT-5.2-Claude-4.5-Recorrected
  Reasoning: The dialogue is high-quality and contextually coherent, but P4's tactic annotation is objectively incorrect. P4's utterance is an active adversarial reframing that shifts focus away from the passing team, not a defensive deflection. This requires correction to [Selective/Framing | Adversarial] with an appropriate tactic like 'Strategic omission'.

  [DIALOGUE UNCHANGED]
  (dialogue identical — only matrix annotation changed)

  [MATRIX CHANGED]

  ── OLD MATRIX ──
    {
      "P2": {"row":"Transparent","col":"Adversarial","tactic":"Misleading emphasis","scale":"Mach-IV","level":"High"},
      "P5": {"row":"Selective / Framing","col":"Cooperative","tactic":"Bayesian hedging","scale":"GRS","level":"High"},
      "P

In [23]:
# ── Recheck Statistics ──────────────────────────────────────────────────────
_recheck_col = [cr.get('Recheck_4c', '') for cr in criteria_records]

_accepted_rows    = [cr['ID'] for cr in criteria_records if cr.get('Recheck_4c') == 'ACCEPTED']
_recorrected_rows = [cr['ID'] for cr in criteria_records if cr.get('Recheck_4c') == 'RECORRECTED']
_regenerated_rows = [cr['ID'] for cr in criteria_records if cr.get('Recheck_4c') == 'REGENERATED']
_api_error_rows   = [cr['ID'] for cr in criteria_records if cr.get('Recheck_4c') == 'API_ERROR']
_unchanged_rows   = [cr['ID'] for cr in criteria_records if cr.get('Recheck_4c') == '']

total_rechecked = len(_accepted_rows) + len(_recorrected_rows) + len(_regenerated_rows) + len(_api_error_rows)

print("=" * 60)
print("  RECHECK 4c STATISTICS")
print("=" * 60)
print(f"  Total rows in criteria CSV:   {len(criteria_records)}")
print(f"  Rows rechecked (Targeted/FC): {total_rechecked}")
print(f"  Rows not rechecked (Accepted/None): {len(_unchanged_rows)}")
print()
print(f"  {'ACCEPTED':<14} {len(_accepted_rows):>3}  (5/5 — no change needed)")
print(f"  {'RECORRECTED':<14} {len(_recorrected_rows):>3}  (4/5 — dialogue/matrix updated)")
print(f"  {'REGENERATED':<14} {len(_regenerated_rows):>3}  (≤3/5 — fully regenerated)")
print(f"  {'API_ERROR':<14} {len(_api_error_rows):>3}  (call failed — unchanged)")
print()

if _recorrected_rows:
    print("  Recorrected game IDs:")
    for gid in _recorrected_rows:
        cr = next((c for c in criteria_records if c['ID'] == gid), {})
        print(f"    {gid:<12}  score: {cr.get('Recheck_4c_Score','')}  LLM: {cr.get('Selected_LLM','')}")

if _regenerated_rows:
    print()
    print("  Regenerated game IDs:")
    for gid in _regenerated_rows:
        cr = next((c for c in criteria_records if c['ID'] == gid), {})
        print(f"    {gid:<12}  score: {cr.get('Recheck_4c_Score','')}  LLM: {cr.get('Selected_LLM','')}")

if _api_error_rows:
    print()
    print("  API errors (manually review):")
    for gid in _api_error_rows:
        print(f"    {gid}")

print("=" * 60)

  RECHECK 4c STATISTICS
  Total rows in criteria CSV:   225
  Rows rechecked (Targeted/FC): 43
  Rows not rechecked (Accepted/None): 0

  ACCEPTED        21  (5/5 — no change needed)
  RECORRECTED     19  (4/5 — dialogue/matrix updated)
  REGENERATED      3  (≤3/5 — fully regenerated)
  API_ERROR        0  (call failed — unchanged)

  Recorrected game IDs:
    G035/1        score: 4 of 5  LLM: GPT-5.2-Claude-4.5-Recorrected
    G055/1        score: 4 of 5  LLM: GPT-5.2-Claude-4.5-Recorrected
    G064/1        score: 4 of 5  LLM: GPT-5.2-Claude-4.5-Recorrected
    G065/1        score: 4 of 5  LLM: GPT-5.2-Claude-4.5-Recorrected
    G097/1        score: 4 of 5  LLM: GPT-5.2-Claude-4.5-Recorrected
    G099/1        score: 4 of 5  LLM: GPT-5.2-Claude-4.5-Recorrected
    G126/1        score: 4 of 5  LLM: GPT-5.2-Claude-4.5-Recorrected
    G129/1        score: 4 of 5  LLM: GPT-5.2-Claude-4.5-Recorrected
    G133/1        score: 4 of 5  LLM: GPT-5.2-Claude-4.5-Recorrected
    G150/1        sc

## Save Updated CSVs

In [25]:
# Save updated Datasets/verified/verified_r1_seeds_combined.csv
out_combined = pd.DataFrame(verified_records)
out_combined.to_csv(OUTPUT_FILE_COMBINED, index=False)
print(f"✓ Saved {OUTPUT_FILE_COMBINED}  ({len(out_combined)} rows)")

# Save updated Datasets/verified/verified_r1_criteria_scores.csv (with Recheck_4c_* columns)
out_criteria = pd.DataFrame(criteria_records)
out_criteria.to_csv(OUTPUT_FILE_CRITERIA, index=False)
print(f"✓ Saved {OUTPUT_FILE_CRITERIA}  ({len(out_criteria)} rows)")

# Summary of Recheck_4c column distribution
if 'Recheck_4c' in out_criteria.columns:
    rc_counts = out_criteria['Recheck_4c'].value_counts(dropna=False)
    print("\nRecheck_4c distribution:")
    for label, count in rc_counts.items():
        print(f"  {label}: {count}")

# Summary of LLM_used suffixes in combined
print("\nLLM_used distribution (updated):")
for llm, count in out_combined['LLM_used'].value_counts().items():
    print(f"  {llm}: {count}")

✓ Saved verified_r1_seeds_combined.csv  (225 rows)
✓ Saved verified_r1_criteria_scores.csv  (225 rows)

Recheck_4c distribution:
  nan: 182
  ACCEPTED: 21
  RECORRECTED: 19
  REGENERATED: 3

LLM_used distribution (updated):
  GPT-5.2: 158
  Gemini-3.1: 24
  GPT-5.2-Claude-4.5-Recorrected: 18
  GPT-5.2-Claude-4.5: 12
  Gemini-3.1-Claude-4.5: 9
  Claude-4.5-Regenerated: 2
  GPT-5.2-Claude-4.5-Regenerated: 1
  Gemini-3.1-Claude-4.5-Recorrected: 1
